# 04 · Instance, Class & Static Methods

**Module 06 – Object-Oriented Programming** · Estimated time: 35 min

> 📘 From the handbook's *Static and Class Methods* chapter, extended. Not covered in the 📗 bootcamp.

### 🎯 What you will learn
- The three kinds of methods and what their first parameter is
- `@classmethod`: working with the class; **alternative constructors** (factory methods)
- `@staticmethod`: utility functions grouped inside a class
- How class methods behave with inheritance
- A decision table

## 1. The three types

| Type | Decorator | First parameter | Can access |
|---|---|---|---|
| Instance method | (none) | `self` (the object) | instance **and** class data |
| Class method | `@classmethod` | `cls` (the class) | class data; can create instances |
| Static method | `@staticmethod` | (none) | nothing automatically: a plain function in the class namespace |

## 2. Instance methods (the default)

In [1]:
class Dog:
    def __init__(self, name):
        self.name = name              # instance attribute

    def speak(self):                  # instance method
        return f"{self.name} says Woof!"

dog = Dog("Buddy")
print(dog.speak())

Buddy says Woof!


## 3. Class methods: `@classmethod`

Receive the **class** as `cls`. Common uses:
1. Reading or modifying **class-level** state (affects all instances)
2. **Factory methods**: alternative constructors

In [2]:
class Animal:
    species = "Mammal"                # class attribute

    @classmethod
    def set_species(cls, new_species):
        cls.species = new_species     # modifies the class attribute

    @classmethod
    def get_species(cls):
        return cls.species

print(Animal.get_species())
Animal.set_species("Reptile")
print(Animal.get_species())
a = Animal()
print(a.get_species())               # can also be called on an instance (less common)

Mammal
Reptile
Reptile


### Alternative constructors (📘)

`__init__` gives you one way to build objects. Class methods give you more, with descriptive names. You've already used them: `dict.fromkeys()`, `datetime.fromisoformat()`, `int.from_bytes()`.

In [3]:
from datetime import date

class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_string(cls, data):           # "Alice-30"
        name, age = data.split("-")
        return cls(name, int(age))        # cls(...) creates a new instance

    @classmethod
    def from_birth_year(cls, name, year):
        return cls(name, date.today().year - year)

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age})"

p = Person.from_string("Alice-30")
print(p.name, p.age)
print(Person.from_birth_year("Ravi", 1995))

Alice 30
Person('Ravi', 31)


### Why `cls` instead of hard-coding `Person`? Inheritance!

In [4]:
class Student(Person):
    pass

s = Student.from_string("Meena-21")
print(s, type(s))                 # a Student, not a Person: cls was Student

Student('Meena', 21) <class '__main__.Student'>


### Counting instances with a class method

In [5]:
class Robot:
    population = 0

    def __init__(self, name):
        self.name = name
        Robot.population += 1

    @classmethod
    def how_many(cls):
        return f"We have {cls.population} robots."

r1, r2 = Robot("R2-D2"), Robot("C-3PO")
print(Robot.how_many())

We have 2 robots.


## 4. Static methods: `@staticmethod`

No `self`, no `cls`. A regular function that **logically belongs** to the class (utility, validation, conversion).

In [6]:
class MathUtils:
    @staticmethod
    def add(a, b):
        return a + b

    @staticmethod
    def is_even(n):
        return n % 2 == 0

print(MathUtils.add(3, 5))
m = MathUtils()
print(m.add(4, 5))               # can also be called on an instance
print(MathUtils.is_even(10))

8
9
True


In [7]:
class Temperature:
    def __init__(self, celsius):
        if not Temperature.is_valid(celsius):
            raise ValueError("Below absolute zero")
        self.celsius = celsius

    @staticmethod
    def is_valid(celsius):
        return celsius >= -273.15

    @staticmethod
    def c_to_f(c):
        return c * 9 / 5 + 32

print(Temperature.c_to_f(100), Temperature.is_valid(-300))

212.0 False


## 5. Putting all three together

In [8]:
class Pizza:
    base_price = 200

    def __init__(self, toppings):
        self.toppings = toppings

    def price(self):                                   # instance method
        return self.base_price + 40 * len(self.toppings)

    @classmethod
    def margherita(cls):                               # factory
        return cls(["mozzarella", "tomato"])

    @classmethod
    def change_base_price(cls, value):                 # class state
        cls.base_price = value

    @staticmethod
    def valid_topping(t):                              # utility
        return t.lower() not in {"pineapple"}

p = Pizza.margherita()
print(p.toppings, p.price())
Pizza.change_base_price(250)
print(p.price(), Pizza.valid_topping("Pineapple"))

['mozzarella', 'tomato'] 280
330 False


## 6. Key differences (📘 table)

| Method type | Needs `self`? | Needs `cls`? | Access instance attrs? | Modify class attrs? |
|---|---|---|---|---|
| Instance | ✅ | ❌ | ✅ | ✅ (via `type(self)` / class name) |
| Class | ❌ | ✅ | ❌ | ✅ |
| Static | ❌ | ❌ | ❌ | ❌ |

### Which one should I use?
- Needs the object's data? → **instance method**
- Needs the class (factory, class state, works with subclasses)? → **class method**
- Needs neither, but belongs with the class conceptually? → **static method** (or simply a module-level function)

## 📝 Summary
- Instance methods get `self`, class methods get `cls`, static methods get nothing.
- `@classmethod` is ideal for alternative constructors; using `cls(...)` supports subclasses.
- `@staticmethod` groups helper functions inside a class.

## 🏋️ Practice
1. Add a `Date.from_iso("2026-10-20")` class method to a `Date(year, month, day)` class.
2. Add a static method `Date.is_leap(year)`.
3. Create `Pizza.vegetarian()` and `Pizza.pepperoni()` factory methods.

<details><summary>Show solutions</summary>

```python
class Date:
    def __init__(self, year, month, day):
        self.year, self.month, self.day = year, month, day
    @classmethod
    def from_iso(cls, s):
        y, m, d = map(int, s.split("-"))
        return cls(y, m, d)
    @staticmethod
    def is_leap(y):
        return (y % 4 == 0 and y % 100 != 0) or y % 400 == 0

class Pizza:
    def __init__(self, toppings): self.toppings = toppings
    @classmethod
    def vegetarian(cls): return cls(["capsicum", "onion", "olives"])
    @classmethod
    def pepperoni(cls): return cls(["pepperoni", "cheese"])
```
</details>